# Word Embeddings -- Word2Vec

A word is defined by it's companies.

Train a shallow net on that idea and geometry falls out.

## Problem Definition

TF-IDF knows "dog" and "puppy" are difference words, but doesn't know they mean nearly the same thing.


Desired Repersentation:

* "dog" and "puppy" land close together in space
* "king - man + woman" land near "queen"

## Basic Concept

Word2Vec, two flavors exploiting that data:

* Skip-gram. Given a center word, predict the surrounding words, cat --> (the, sat, on) with window size 2.
* CBOW(continous bag of words). Given surrounding words, predict the center, (the, sat, on) --> cat.

**Skip-gram is slower to train but handles rate words better, it became the default.**

## Build your Own

## Trainning pairs from a corpus

In [1]:
def skipgram_pairs(docs, window=2):
    pairs = []
    for doc in docs:
        for i, center in enumerate(doc):
            for j in range(max(0, 1 - window), min(len(doc), i + window + 1)):
                if i == j:
                    continue
                pairs.append((center, doc[j]))

    return pairs

skipgram_pairs([["the", "cat", "sat", "on", "the", "mat"]], window=2)

[('the', 'cat'),
 ('the', 'sat'),
 ('cat', 'the'),
 ('cat', 'sat'),
 ('cat', 'on'),
 ('sat', 'the'),
 ('sat', 'cat'),
 ('sat', 'on'),
 ('sat', 'the'),
 ('on', 'the'),
 ('on', 'cat'),
 ('on', 'sat'),
 ('on', 'the'),
 ('on', 'mat'),
 ('the', 'the'),
 ('the', 'cat'),
 ('the', 'sat'),
 ('the', 'on'),
 ('the', 'mat'),
 ('mat', 'the'),
 ('mat', 'cat'),
 ('mat', 'sat'),
 ('mat', 'on'),
 ('mat', 'the')]

## Embedding tables

In [2]:
import numpy as np

def init_embedding(vocab_size, dim, seed=0):
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 0.1, size=(vocab_size, dim))
    W_prime = rng.normal(0, 0.1, size=(vocab_size, dim))
    return W, W_prime

## Negative sampling object

### Loss and gradients (SGNS)

对一个中心词 $v_c$、正样本上下文 $u_{+}$、负样本 $\{u_k\}_{k=1}^{K}$，负采样目标（越大越好；训练时最小化其相反数）为：

$$
\mathcal{L}
= \log\sigma(v_c^{\top} u_{+})
+ \sum_{k=1}^{K}\log\sigma(-v_c^{\top} u_k)
$$

其中 $\sigma(x)=1/(1+e^{-x})$。记 $s_{+}=\sigma(v_c^{\top} u_{+})$，$s_k=\sigma(v_c^{\top} u_k)$。

对点积的常用性质：

$$
\frac{\partial}{\partial x}\log\sigma(x)=1-\sigma(x),\qquad
\frac{\partial}{\partial x}\log\sigma(-x)=-\sigma(x)
$$

因此最小化 $-\mathcal{L}$ 时的梯度（与代码里的 SGD 更新一致）：

$$
\begin{aligned}
\nabla_{v_c}(-\mathcal{L})
&= (s_{+}-1)\,u_{+} + \sum_{k} s_k\,u_k \\
\nabla_{u_{+}}(-\mathcal{L})
&= (s_{+}-1)\,v_c \\
\nabla_{u_k}(-\mathcal{L})
&= s_k\,v_c
\end{aligned}
$$

直观：$s_{+}\to 1$、$s_k\to 0$ 时梯度都趋近 0；否则把 $v_c$ 往 $u_{+}$ 拉、往 $u_k$ 推。


In [3]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20, 20)))


def train_pair(W, W_prime, center_idx, context_idx, negative_indices, lr):
    """
    Skip-gram + Negative Sampling 的一步 SGD。
    目标：拉近 (中心词, 正样本上下文)，推远 (中心词, 负采样词)。
    W: 中心词表；W_prime: 上下文/负样本词表（两套向量是 Word2Vec 惯例）。
    """
    # 取出当前涉及的向量
    v_c = W[center_idx]                 # 中心词向量
    u_pos = W_prime[context_idx]        # 正样本（真实窗口内的上下文）
    u_negs = W_prime[negative_indices]  # 负样本们（随机抽的“假上下文”）

    # σ(v·u)：把点积变成 (0,1)，可理解为“是上下文”的概率
    pos_score = sigmoid(v_c @ u_pos)         # 希望 → 1
    neg_scores = sigmoid(u_negs @ v_c)       # 希望 → 0

    # dL/dv_c：正样本项 (σ-1)*u_pos，负样本项 σ*u_neg（来自 -logσ(-·) 的梯度）
    grad_center = (pos_score - 1) * u_pos
    for i, u in enumerate(u_negs):
        grad_center += neg_scores[i] * u

    # 更新上下文表 W'（中心表 W 只更新 center）
    W_prime[context_idx] -= lr * (pos_score - 1) * v_c
    for i, neg_idx in enumerate(negative_indices):
        W_prime[neg_idx] -= lr * neg_scores[i] * v_c
    W[center_idx] -= lr * grad_center

    pos_score = sigmoid(v_c @ u_pos)
    neg_scores = sigmoid(u_negs @ v_c)

    grad_center = (pos_score - 1) * u_pos
    for i, u in enumerate(u_negs):
        grad_center += neg_scores[i] * u

    W[context_idx] = W[context_idx]
    W_prime[context_idx] -= lr * (pos_score - 1) * v_c
    for i, neg_idx in enumerate(negative_indices):
        W_prime[neg_idx] -= lr * neg_scores[i] * v_c
    W[center_idx] -= lr * grad_center


## Training

In [ ]:
def build_vocab(docs):
    vocab = {}
    for doc in docs:
        for token in doc:
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab

def train(docs, dim=16, window=2, k_neg=5, epochs=100, lr=0.05, seed=0):
    vocab = build_vocab(docs)
    vocab_size = len(vocab)
    rng = np.random.default_rng(seed)
    W, W_prime = init_embedding(vocab_size, dim, seed)
    pairs = skipgram_pairs(docs, window)

    for epoch in range(epochs):
        rng.shuffle(pairs)
        for center, context in pairs:
            c_idx = vocab[center]
            ctx_idx = vocab[context]
            negs = rng.integers(0, vocab_size, size=k_neg)
            negs = [n for n in negs if n != ctx_idx and n != c_idx]
            train_pair(W, W_prime, c_idx, ctx_idx, negs, lr)